# EMICRON 2023: formalización, brecha digital y crecimiento de ingresos

Este análisis combina los microdatos procesados en `data/processed/emicron_core.parquet` con el módulo TIC original para calcular indicadores ponderados por `fex_c`.

- **Formalización:** promedio ponderado de cuatro componentes: RUT, Cámara de Comercio, registro contable formal (`p640` en 1, 2 o 4, según la definición validada en Fase 4) y declaración de al menos un impuesto.
- **Brecha digital:** proporción ponderada de negocios con acceso a internet (`p2524 = 1`) por departamento, usando el módulo TIC.
- **Crecimiento interanual:** compara el ingreso mensual de 2023 anualizado (mes × 12) con los ingresos totales de 2022 (`ventas_anio_anterior`), usando totales ponderados por `fex_c`.

En `AREA` y `COD_DEPTO` se muestran los códigos del microdato, porque el Parquet y el diccionario disponible no incluyen una tabla de etiquetas geográficas.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

CIUDAD_NOMBRES = {
    "05": "Medellín AM", "08": "Barranquilla AM", "11": "Bogotá",
    "13": "Cartagena", "15": "Tunja", "17": "Manizales AM",
    "18": "Florencia", "19": "Popayán", "20": "Valledupar",
    "23": "Montería", "27": "Quibdó", "41": "Neiva",
    "44": "Riohacha", "47": "Santa Marta", "50": "Villavicencio",
    "52": "Pasto", "54": "Cúcuta AM", "63": "Armenia",
    "66": "Pereira AM", "68": "Bucaramanga AM", "70": "Sincelejo",
    "73": "Ibagué", "76": "Cali AM", "88": "San Andrés",
}

DEPARTAMENTO_NOMBRES = {
    "05": "Antioquia", "08": "Atlántico", "11": "Bogotá", "13": "Bolívar",
    "15": "Boyacá", "17": "Caldas", "18": "Caquetá", "19": "Cauca",
    "20": "Cesar", "23": "Córdoba", "25": "Cundinamarca", "27": "Chocó",
    "41": "Huila", "44": "La Guajira", "47": "Magdalena", "50": "Meta",
    "52": "Nariño", "54": "Norte de Santander", "63": "Quindío",
    "66": "Risaralda", "68": "Santander", "70": "Sucre", "73": "Tolima",
    "76": "Valle del Cauca", "88": "San Andrés",
}

SECTOR_NOMBRES = {
    "01": "Agricultura", "02": "Minería", "03": "Industria manufacturera",
    "04": "Construcción", "05": "Comercio", "06": "Transporte",
    "07": "Alojamiento y comida", "08": "Información y comunicaciones",
    "09": "Inmobiliarias y profesionales", "10": "Educación", "11": "Salud",
    "12": "Artes y entretenimiento",
}

parquet_name = Path("data/processed/emicron_core.parquet")
project_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / parquet_name).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "No se encontró data/processed/emicron_core.parquet desde el directorio actual."
    )

parquet_path = project_root / parquet_name
core = pd.read_parquet(parquet_path)

required_core_columns = {
    "directorio", "secuencia_p", "secuencia_encuesta", "fex_c", "area",
    "cod_depto", "grupos12", "p1633", "p1055", "p640", "p2991",
    "p2992", "p2993", "p3032_1",
}
missing_core_columns = sorted(required_core_columns.difference(core.columns))
if missing_core_columns:
    raise ValueError(
        "El Parquet no contiene columnas necesarias para el portafolio de Omar: "
        + ", ".join(missing_core_columns)
    )

core["fex_c"] = pd.to_numeric(core["fex_c"], errors="coerce")
core = core.loc[core["fex_c"].gt(0)].copy()
print(f"Archivo: {parquet_path.relative_to(project_root)}")
print(f"Registros con factor de expansión positivo: {len(core):,}")

Archivo: data\processed\emicron_core.parquet
Registros con factor de expansión positivo: 81,018


## 1. Formalización por ciudad

El índice es el promedio ponderado de cuatro prácticas formales. Las barras muestran las diez áreas con mayor puntaje; los códigos se conservan tal como están en el microdato.

In [2]:
formal = core[[
    "area", "fex_c", "p1633", "p1055", "p640", "p2991", "p2992", "p2993"
]].copy()
formal["area"] = formal["area"].astype("string").str.strip()
formal = formal.loc[formal["area"].notna() & formal["area"].ne("")].copy()

formal_components = {
    "rut": pd.to_numeric(formal["p1633"], errors="coerce").eq(1),
    "camara": pd.to_numeric(formal["p1055"], errors="coerce").eq(1),
    "contabilidad_formal": pd.to_numeric(formal["p640"], errors="coerce").isin([1, 2, 4]),
    "declaracion_impuestos": pd.concat(
        [pd.to_numeric(formal[column], errors="coerce").eq(1)
         for column in ["p2991", "p2992", "p2993"]],
        axis=1,
    ).any(axis=1),
}
for column, values in formal_components.items():
    formal[column] = values.fillna(False).astype("int8")

formal["indice_formalizacion_pct"] = formal[
    list(formal_components)
].mean(axis=1) * 100
formal["indice_ponderado"] = formal["indice_formalizacion_pct"] * formal["fex_c"]

formalizacion_ciudad = (
    formal.groupby("area", as_index=False)
    .agg(puntaje_ponderado=("indice_ponderado", "sum"), peso_total=("fex_c", "sum"))
)
formalizacion_ciudad["indice_formalizacion_pct"] = (
    formalizacion_ciudad["puntaje_ponderado"] / formalizacion_ciudad["peso_total"]
)
formalizacion_top10 = formalizacion_ciudad.nlargest(10, "indice_formalizacion_pct")
formalizacion_top10 = formalizacion_top10.sort_values(
    "indice_formalizacion_pct", ascending=False
)
formalizacion_top10["ciudad"] = (
    formalizacion_top10["area"].map(CIUDAD_NOMBRES)
    .fillna(formalizacion_top10["area"])
)
formalizacion_top10

,area,puntaje_ponderado,peso_total,indice_formalizacion_pct,ciudad
23,88,9.740280e+04,2941.888273,33.108941,San Andrés
21,73,9.055906e+05,30260.094083,29.926893,Ibagué
4,15,3.716624e+05,13096.029644,28.379780,Tunja
2,11,1.339458e+07,508114.626803,26.361327,Bogotá
17,63,7.644926e+05,29154.944292,26.221713,Armenia
5,17,8.353777e+05,34083.529355,24.509718,Manizales AM
19,68,2.660585e+06,126267.837808,21.070965,Bucaramanga AM
18,66,1.007209e+06,48637.991616,20.708268,Pereira AM
0,05,7.165204e+06,383452.990198,18.686003,Medellín AM
7,19,4.348447e+05,23303.109349,18.660371,Popayán


In [3]:
fig_formalizacion = px.bar(
    formalizacion_top10,
    x="indice_formalizacion_pct",
    y="ciudad",
    orientation="h",
    text="indice_formalizacion_pct",
    title="Top 10 ciudades por índice de formalización",
    labels={
        "ciudad": "Ciudad",
        "indice_formalizacion_pct": "Índice de formalización (%)",
    },
    color="indice_formalizacion_pct",
    color_continuous_scale=px.colors.sequential.Plasma,
    template="plotly_dark",
)
fig_formalizacion.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
)
fig_formalizacion.update_layout(
    title={"x": 0, "xanchor": "left", "font": {"size": 20}},
    font={"family": "Arial, sans-serif", "size": 12},
    xaxis={"title": "Índice de formalización (%)", "range": [0, 105], "showgrid": True, "gridcolor": "#34445A"},
    yaxis={"title": "Ciudad", "categoryorder": "array", "categoryarray": formalizacion_top10["ciudad"].tolist(), "autorange": "reversed"},
    coloraxis_showscale=False,
    margin={"l": 155, "r": 55, "t": 75, "b": 55},
)
fig_formalizacion.show()

## 2. Brecha digital: departamentos con menor acceso a internet

El módulo TIC se une al Parquet mediante la llave compuesta. El porcentaje de acceso se calcula entre las respuestas válidas y pondera cada negocio con el factor `fex_c`; se muestran los cinco departamentos con menor acceso.

In [4]:
tic_path = project_root / "data" / "raw" / "tic.csv"
if not tic_path.is_file():
    raise FileNotFoundError(f"No se encontró el módulo requerido: {tic_path}")

tic = pd.read_csv(
    tic_path,
    dtype={
        "DIRECTORIO": "string",
        "SECUENCIA_P": "string",
        "SECUENCIA_ENCUESTA": "string",
        "COD_DEPTO": "string",
    },
    keep_default_na=True,
)
tic.columns = tic.columns.str.strip().str.lower()
tic_key = ["directorio", "secuencia_p", "secuencia_encuesta"]
required_tic = set(tic_key + ["p2524", "cod_depto"])
missing_tic = sorted(required_tic.difference(tic.columns))
if missing_tic:
    raise ValueError("tic.csv no contiene: " + ", ".join(missing_tic))

factor_por_negocio = core[tic_key + ["fex_c"]].copy()
for column in tic_key:
    tic[column] = pd.to_numeric(tic[column], errors="coerce").astype("Int64")
    factor_por_negocio[column] = pd.to_numeric(
        factor_por_negocio[column], errors="coerce"
    ).astype("Int64")
if factor_por_negocio.duplicated(tic_key).any():
    raise ValueError("Hay llaves compuestas duplicadas en el Parquet.")

tic = tic.merge(
    factor_por_negocio,
    on=tic_key,
    how="inner",
    validate="one_to_one",
)
tic["p2524"] = pd.to_numeric(tic["p2524"], errors="coerce")
tic["cod_depto"] = tic["cod_depto"].astype("string").str.strip()
tic = tic.loc[
    tic["p2524"].notna()
    & tic["cod_depto"].notna()
    & tic["cod_depto"].ne("")
    & tic["fex_c"].gt(0)
].copy()
tic["acceso_ponderado"] = tic["fex_c"].where(tic["p2524"].eq(1), 0)

internet_depto = (
    tic.groupby("cod_depto", as_index=False)
    .agg(acceso_ponderado=("acceso_ponderado", "sum"), peso_total=("fex_c", "sum"))
)
internet_depto["pct_acceso_internet"] = (
    100 * internet_depto["acceso_ponderado"] / internet_depto["peso_total"]
)
internet_depto["departamento"] = (
    internet_depto["cod_depto"].map(DEPARTAMENTO_NOMBRES)
    .fillna(internet_depto["cod_depto"])
)
brecha_top5 = internet_depto.nsmallest(5, "pct_acceso_internet").sort_values(
    "pct_acceso_internet", ascending=True
)
brecha_top5

,cod_depto,acceso_ponderado,peso_total,pct_acceso_internet,departamento
21,70,20451.534996,180333.612104,11.340945,Sucre
16,52,47694.274487,402863.128478,11.838828,Nariño
9,23,45393.141006,278170.292986,16.318472,Córdoba
13,44,13817.232968,83675.718755,16.512835,La Guajira
11,27,7300.923481,36296.866595,20.114473,Chocó


In [5]:
fig_internet = px.bar(
    brecha_top5,
    x="pct_acceso_internet",
    y="departamento",
    orientation="h",
    text="pct_acceso_internet",
    title="Departamentos con menor acceso a internet",
    labels={
        "departamento": "Departamento",
        "pct_acceso_internet": "Acceso a internet (%)",
    },
    color="pct_acceso_internet",
    color_continuous_scale=px.colors.sequential.Turbo,
    template="plotly_dark",
)
fig_internet.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
)
fig_internet.update_layout(
    title={"x": 0, "xanchor": "left", "font": {"size": 20}},
    font={"family": "Arial, sans-serif", "size": 12},
    xaxis={"title": "Acceso a internet (%)", "range": [0, 105], "showgrid": True, "gridcolor": "#34445A"},
    yaxis={"title": "Departamento", "categoryorder": "array", "categoryarray": brecha_top5["departamento"].tolist(), "autorange": "reversed"},
    coloraxis_showscale=False,
    margin={"l": 155, "r": 55, "t": 75, "b": 55},
)
fig_internet.show()

## 3. Crecimiento interanual de ingresos por departamento

Se anualiza el ingreso mensual de 2023 (`ventas_mes_anterior × 12`) y se compara con el total de ingresos de 2022 (`ventas_anio_anterior`). La variación se calcula a partir de los totales ponderados por `fex_c`; el gráfico muestra los diez departamentos con mayor crecimiento.

In [6]:
required_growth_columns = {
    "cod_depto", "fex_c", "ventas_mes_anterior", "ventas_anio_anterior"
}
missing_growth_columns = sorted(required_growth_columns.difference(core.columns))
if missing_growth_columns:
    raise ValueError(
        "El Parquet no contiene columnas necesarias para el crecimiento interanual: "
        + ", ".join(missing_growth_columns)
    )

growth = core[
    ["cod_depto", "fex_c", "ventas_mes_anterior", "ventas_anio_anterior"]
].copy()
growth["cod_depto"] = growth["cod_depto"].astype("string").str.strip()
for column in ["ventas_mes_anterior", "ventas_anio_anterior"]:
    growth[column] = pd.to_numeric(growth[column], errors="coerce")

growth = growth.loc[
    growth["cod_depto"].notna()
    & growth["cod_depto"].ne("")
    & growth["fex_c"].gt(0)
    & growth["ventas_mes_anterior"].notna()
    & growth["ventas_anio_anterior"].gt(0)
].copy()
growth["ingresos_2023_anualizados_ponderados"] = (
    growth["ventas_mes_anterior"] * 12 * growth["fex_c"]
)
growth["ingresos_2022_ponderados"] = (
    growth["ventas_anio_anterior"] * growth["fex_c"]
)

growth_by_department = (
    growth.groupby("cod_depto", as_index=False)
    .agg(
        ingresos_2023_ponderados=("ingresos_2023_anualizados_ponderados", "sum"),
        ingresos_2022_ponderados=("ingresos_2022_ponderados", "sum"),
        muestra=("fex_c", "size"),
    )
)
growth_by_department["variacion_interanual_pct"] = (
    100
    * (growth_by_department["ingresos_2023_ponderados"]
       - growth_by_department["ingresos_2022_ponderados"])
    / growth_by_department["ingresos_2022_ponderados"]
)
top10_crecimiento = growth_by_department.nlargest(
    10, "variacion_interanual_pct"
).sort_values("variacion_interanual_pct", ascending=False)
top10_crecimiento["departamento"] = (
    top10_crecimiento["cod_depto"].map(DEPARTAMENTO_NOMBRES)
    .fillna(top10_crecimiento["cod_depto"])
)
top10_crecimiento

,cod_depto,ingresos_2023_ponderados,ingresos_2022_ponderados,muestra,variacion_interanual_pct,departamento
7,19,3.562658e+12,2.402157e+12,2629,48.310813,Cauca
4,15,7.690310e+12,5.392847e+12,2088,42.602037,Boyacá
11,27,6.738868e+11,5.143324e+11,1787,31.021659,Chocó
9,23,5.135475e+12,3.926956e+12,3104,30.774960,Córdoba
16,52,7.812044e+12,6.048271e+12,4765,29.161615,Nariño
17,54,5.209004e+12,4.034679e+12,2331,29.105759,Norte de Santander
24,88,1.722545e+11,1.343692e+11,629,28.194951,San Andrés
12,41,3.325418e+12,2.640968e+12,2070,25.916639,Huila
8,20,4.234990e+12,3.523959e+12,2900,20.177043,Cesar
22,73,5.123623e+12,4.280540e+12,2062,19.695724,Tolima


In [7]:
fig_crecimiento = px.bar(
    top10_crecimiento,
    x="variacion_interanual_pct",
    y="departamento",
    orientation="h",
    text="variacion_interanual_pct",
    title="Departamentos con mayor crecimiento interanual de ingresos",
    labels={
        "departamento": "Departamento",
        "cod_depto": "Código de departamento",
        "variacion_interanual_pct": "Crecimiento interanual (%)",
        "ingresos_2023_ponderados": "Ingresos 2023 anualizados y ponderados",
        "ingresos_2022_ponderados": "Ingresos 2022 ponderados",
        "muestra": "Registros de la muestra",
    },
    color="variacion_interanual_pct",
    color_continuous_scale=px.colors.sequential.Viridis,
    template="plotly_dark",
    hover_data={
        "cod_depto": True,
        "ingresos_2023_ponderados": ":,.0f",
        "ingresos_2022_ponderados": ":,.0f",
        "muestra": True,
    },
)
fig_crecimiento.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
)
fig_crecimiento.update_layout(
    title={"x": 0, "xanchor": "left", "font": {"size": 20}},
    font={"family": "Arial, sans-serif", "size": 12},
    xaxis={"title": "Crecimiento interanual (%)", "rangemode": "tozero", "showgrid": True, "gridcolor": "#34445A"},
    yaxis={"title": "Departamento", "categoryorder": "array", "categoryarray": top10_crecimiento["departamento"].tolist(), "autorange": "reversed"},
    coloraxis_showscale=False,
    margin={"l": 155, "r": 55, "t": 75, "b": 55},
)
fig_crecimiento.show()